<a href="https://colab.research.google.com/github/ArmandoArV/IntroDataScienceProyecto/blob/master/MainNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis Global de Precios de Combustibles 2020–2026
## Evolución, Volatilidad y Factores Explicativos en 84 Países

**Autores:** Armando Arredondo, Bastián Hernández, Francisco Nahamias, Eduardo Albornoz  
**Institución:** Universidad Católica de Chile  
**Curso:** IMT-3860 — Introducción a Data Science  
**Docente:** Alejandro Cataldo  

---

Este cuaderno consolida el análisis completo del panel semanal de precios de combustibles para 84 países (2020–2026). Integra fuentes externas de energía y riesgo geopolítico, cubre limpieza, ingeniería de variables, análisis estadístico, modelos predictivos y análisis de crisis.

## 0. Setup
### 0.1 Ruta a los datos

In [ ]:
import os
from pathlib import Path

_candidates = [
    Path('../../Datasets'),
    Path('../Datasets'),
    Path('Datasets'),
    Path('../../../Datasets'),
]
DATA_DIR = None
for p in _candidates:
    if p.is_dir():
        DATA_DIR = p.resolve()
        break
if DATA_DIR is None:
    raise FileNotFoundError('No se encontro la carpeta Datasets.')

FILES = {
    'main':  'global_fuel_prices_2020_2026.csv',
    'brent': 'Brend Europa Fred.csv',
    'ovx':   'CBOE Crude Oil ETF Volatility.csv',
    'dxy':   'Nominal Broand US Dollar.csv',
    'gpr':   'data_gpr_export(Sheet1).csv',
}
FRED_FILES_EXTRA = {
    'rbob_ny':       ('DGASNYH.csv',                 'csv_fred', 'DGASNYH'),
    'wti':           ('DCOILWTICO.csv',              'csv_fred', 'DCOILWTICO'),
    'henryhub':      ('DHHNGSP.csv',                 'csv_fred', 'DHHNGSP'),
    'inv_crude':     ('WCESTUS1w.xls',               'xls_eia',  None),
    'inv_gasoline':  ('Stock_of_total_gasoline.xls', 'xls_eia',  None),
    'refinery_util': ('Utilizacion_of_refinery.xls', 'xls_eia', None),
}

INCOME_ORDER  = ['Low', 'Middle', 'High']
SUBSIDY_ORDER = ['Low', 'Medium', 'High', 'Very High']

print(f'DATA_DIR: {DATA_DIR}')
for tag, fname in FILES.items():
    status = 'OK' if (DATA_DIR / fname).exists() else 'FALTA'
    print(f'  [{status}] {fname}')

### 0.2 Instalación de dependencias

In [ ]:
import importlib, subprocess, sys

required = {'statsmodels':'statsmodels','linearmodels':'linearmodels',
            'xlrd':'xlrd','openpyxl':'openpyxl','itables':'itables','plotly':'plotly'}
missing = [pkg for mod, pkg in required.items()
           if not importlib.util.find_spec(mod)]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    print('Instalados:', missing)
else:
    print('Todas las dependencias disponibles.')

### 0.3 Imports y configuración

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import levene, mannwhitneyu, norm
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
from linearmodels.panel import PanelOLS
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from itables import show

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

ITABLE_KW   = dict(maxBytes=0, scrollX=True)
COLOR_SEQ   = px.colors.qualitative.Set2
TEMPLATE    = 'plotly_white'

# Crisis periods
COVID_START   = pd.Timestamp('2020-03-11')
COVID_END     = pd.Timestamp('2021-05-31')
UKRAINE_START = pd.Timestamp('2022-02-24')
UKRAINE_END   = pd.Timestamp('2023-03-31')

def add_crisis_bands(fig):
    for start, end, label, color in [
        (COVID_START,   COVID_END,   'COVID-19',      'rgba(30,144,255,0.10)'),
        (UKRAINE_START, UKRAINE_END, 'Guerra Ucrania','rgba(220,50,50,0.10)'),
    ]:
        fig.add_vrect(x0=start, x1=end, fillcolor=color, line_width=0,
                      annotation_text=label, annotation_position='top left',
                      annotation_font_size=10)
    return fig

print('Librerias cargadas.')

## 1. Ingesta y Armonización de Datos
### 1.1 Dataset principal

In [ ]:
def load_main(data_dir):
    df = pd.read_csv(data_dir / FILES['main'], parse_dates=['date'])
    df['country']      = df['country'].astype('category')
    df['region']       = df['region'].astype('category')
    df['income_level'] = pd.Categorical(df['income_level'],  categories=INCOME_ORDER,  ordered=True)
    df['subsidy_level']= pd.Categorical(df['subsidy_level'], categories=SUBSIDY_ORDER, ordered=True)
    return df.sort_values(['country','date']).reset_index(drop=True)

df_main = load_main(DATA_DIR)
print(f'Filas: {df_main.shape[0]:,}  |  Columnas: {df_main.shape[1]}')
print(f'Rango temporal: {df_main.date.min().date()} -> {df_main.date.max().date()}')
print(f'Paises: {df_main.country.nunique()}  |  Regiones: {df_main.region.nunique()}')
show(df_main.head(10), **ITABLE_KW)

### 1.2 Tipos y categorías

In [ ]:
print('Tipos de datos:')
print(df_main.dtypes.to_string())
print(f'Memoria: {df_main.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

for col in ['region','income_level','subsidy_level']:
    vals = df_main[col].cat.categories.tolist()
    print(f'  {col}: {vals}')

### 1.3 Series FRED externas (Brent, OVX, DXY, GPR)

In [ ]:
def load_fred(data_dir, file_key, value_col, new_name):
    path = data_dir / FILES[file_key]
    df = pd.read_csv(path, parse_dates=['observation_date'])
    df = df.rename(columns={'observation_date':'date', value_col: new_name})
    df[new_name] = pd.to_numeric(df[new_name], errors='coerce')
    return df.dropna().sort_values('date').reset_index(drop=True)

def load_gpr(data_dir):
    path = data_dir / FILES['gpr']
    raw = pd.read_csv(path, sep=';', encoding='utf-8-sig')
    raw['month'] = pd.to_datetime(raw['month'], format='%d/%m/%Y', errors='coerce')
    out = raw[['month','GPR','GPRA','GPRT']].copy()
    for c in ['GPR','GPRA','GPRT']:
        out[c] = pd.to_numeric(out[c].astype(str).str.replace(',','.'), errors='coerce')
    return out.dropna(subset=['month']).sort_values('month').reset_index(drop=True)                  .rename(columns={'month':'date','GPR':'gpr','GPRA':'gpr_acts','GPRT':'gpr_threats'})

df_brent = load_fred(DATA_DIR, 'brent', 'DCOILBRENTEU', 'brent_fred')
df_ovx   = load_fred(DATA_DIR, 'ovx',   'OVXCLS',       'ovx')
df_dxy   = load_fred(DATA_DIR, 'dxy',   'DTWEXBGS',     'dxy')
df_gpr   = load_gpr(DATA_DIR)

print(f'Brent: {len(df_brent)} obs diarias')
print(f'OVX  : {len(df_ovx)} obs diarias')
print(f'DXY  : {len(df_dxy)} obs diarias')
print(f'GPR  : {len(df_gpr)} obs mensuales')

### 1.4 Series adicionales (RBOB, WTI, inventarios, refinería)

In [ ]:
def load_csv_fred_local(path, value_col, alias):
    df = pd.read_csv(path, parse_dates=['observation_date'])
    df = df.rename(columns={'observation_date':'date', value_col: alias})
    df[alias] = pd.to_numeric(df[alias], errors='coerce')
    return df.dropna(subset=[alias]).sort_values('date').reset_index(drop=True)

def load_xls_eia_local(path, alias):
    df = pd.read_excel(path, sheet_name='Data 1', skiprows=2, names=['date', alias])
    df['date']  = pd.to_datetime(df['date'], errors='coerce')
    df[alias]   = pd.to_numeric(df[alias],   errors='coerce')
    return df.dropna(subset=['date', alias]).sort_values('date').reset_index(drop=True)

fred_extra = {}
for alias, (fname, tipo, value_col) in FRED_FILES_EXTRA.items():
    full_path = DATA_DIR / fname
    if not full_path.exists():
        print(f'  [FALTA] {fname}')
        fred_extra[alias] = None
        continue
    try:
        if tipo == 'csv_fred':
            fred_extra[alias] = load_csv_fred_local(full_path, value_col, alias)
        else:
            fred_extra[alias] = load_xls_eia_local(full_path, alias)
        print(f'  [OK] {alias}: {len(fred_extra[alias])} obs')
    except Exception as e:
        print(f'  [ERROR] {alias}: {e}')
        fred_extra[alias] = None

### 1.5 Merge al panel semanal

In [ ]:
def to_weekly_mean(df_daily, value_cols):
    out = df_daily.copy()
    out['week'] = out['date'].dt.to_period('W-MON').dt.start_time
    return out.groupby('week')[value_cols].mean().reset_index().rename(columns={'week':'date'})

def align_to_panel(df_weekly, panel_dates, value_cols):
    serie = df_weekly.set_index('date').sort_index()
    all_dates = serie.index.union(panel_dates)
    result = (serie.reindex(all_dates)
                   .interpolate(method='time', limit=14)
                   .ffill().bfill()
                   .loc[panel_dates]
                   .reset_index().rename(columns={'index':'date'}))
    return result

def expand_monthly(df_monthly, panel_dates, value_cols):
    daily_grid = pd.date_range(panel_dates.min(), panel_dates.max(), freq='D')
    expanded = (df_monthly.set_index('date')[value_cols]
                          .reindex(daily_grid, method='ffill').bfill()
                          .reset_index().rename(columns={'index':'date'}))
    return expanded[expanded['date'].isin(panel_dates)].reset_index(drop=True)

panel_dates = pd.DatetimeIndex(sorted(df_main['date'].unique()))

brent_w = align_to_panel(to_weekly_mean(df_brent, ['brent_fred']), panel_dates, ['brent_fred'])
ovx_w   = align_to_panel(to_weekly_mean(df_ovx,   ['ovx']),        panel_dates, ['ovx'])
dxy_w   = align_to_panel(to_weekly_mean(df_dxy,   ['dxy']),        panel_dates, ['dxy'])
gpr_w   = expand_monthly(df_gpr, panel_dates, ['gpr','gpr_acts','gpr_threats'])

def safe_merge(left, right, on='date', name=''):
    dups = right[right.duplicated(subset=[on], keep=False)]
    if len(dups):
        right = right.groupby(on).mean(numeric_only=True).reset_index()
    result = left.merge(right, on=on, how='left', validate='many_to_one')
    print(f'  {name}: {result.shape[0]:,} filas x {result.shape[1]} cols')
    return result

print('Merging fuentes externas...')
df = safe_merge(df_main, brent_w,  name='Brent FRED')
df = safe_merge(df,      ovx_w,    name='OVX')
df = safe_merge(df,      dxy_w,    name='DXY')
df = safe_merge(df,      gpr_w,    name='GPR')

for alias, df_src in fred_extra.items():
    if df_src is None:
        df[alias] = np.nan
        continue
    freq_days = (df_src['date'].diff().median()).days if len(df_src) > 1 else 1
    src_w = to_weekly_mean(df_src, [alias]) if freq_days <= 2 else df_src.copy()
    src_a = align_to_panel(src_w, panel_dates, [alias])
    df = safe_merge(df, src_a, name=alias)

print(f'Panel final: {df.shape[0]:,} filas x {df.shape[1]} cols')
show(df.head(10), **ITABLE_KW)

## 2. Calidad de Datos y Limpieza
### 2.1 Reporte de valores faltantes

In [ ]:
missing_report = (
    pd.DataFrame({
        'columna':       df.columns,
        'dtype':         [str(df[c].dtype) for c in df.columns],
        'faltantes':     df.isna().sum().values,
        'pct_faltantes': (df.isna().mean() * 100).round(2).values,
        'unicos':        [df[c].nunique(dropna=True) for c in df.columns],
    })
    .assign(estado=lambda x: np.where(x['faltantes'] == 0, 'Completo', 'Revisar'))
    .sort_values('faltantes', ascending=False)
    .reset_index(drop=True)
)
show(missing_report, **ITABLE_KW)

### 2.2 Precios negativos o cercanos a cero

In [ ]:
muy_bajo = df[df['petrol_usd_liter'] < 0.10]
print(f'Observaciones con gasolina < 0.10 USD/L: {len(muy_bajo):,}')
if len(muy_bajo):
    show(muy_bajo[['date','country','petrol_usd_liter']].value_counts('country')
           .reset_index().rename(columns={0:'n'}), **ITABLE_KW)

### 2.3 Detección de outliers con z-score robusto (MAD)

In [ ]:
def robust_z_scores(df, target='petrol_usd_liter', threshold=5.0):
    out = []
    for country, grp in df.groupby('country', observed=True):
        median = grp[target].median()
        mad    = (grp[target] - median).abs().median()
        if mad == 0:
            continue
        z      = 0.6745 * (grp[target] - median) / mad
        anom   = grp[z.abs() > threshold].copy()
        anom['z_score'] = z[z.abs() > threshold]
        out.append(anom)
    return pd.concat(out) if out else pd.DataFrame()

anom = robust_z_scores(df)
print(f'Outliers detectados (z-MAD > 5): {len(anom):,}')
print(f'Paises afectados: {anom["country"].nunique()}')

if len(anom):
    show(anom[['date','country','petrol_usd_liter','z_score']].sort_values('z_score', ascending=False).head(20),
         **ITABLE_KW)

### 2.4 Visualización de outliers

In [ ]:
fuel_cols = ['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter']
fuel_labels = ['Gasolina', 'Diesel', 'GLP']

fig = make_subplots(rows=1, cols=3, subplot_titles=fuel_labels)
for i, (col, lbl) in enumerate(zip(fuel_cols, fuel_labels), 1):
    fig.add_trace(go.Box(y=df[col].dropna(), name=lbl,
                         marker_color=COLOR_SEQ[i-1], boxpoints='outliers',
                         jitter=0.3, pointpos=-1.8), row=1, col=i)
fig.update_layout(template=TEMPLATE, title='Distribucion de precios retail — outliers marcados',
                  showlegend=False, height=420)
fig.show()

### 2.5 Retornos semanales extremos

In [ ]:
df_sorted = df.sort_values(['country','date'])
df['log_petrol'] = np.log(df['petrol_usd_liter'].clip(lower=1e-3))
df['dlog_petrol_raw'] = df.groupby('country', observed=True)['log_petrol'].diff()

ext = df[df['dlog_petrol_raw'].abs() > 0.15]
print(f'Retornos extremos (|dlog| > 0.15): {len(ext):,}')

top_paises = ext['country'].value_counts().head(10).reset_index()
top_paises.columns = ['pais', 'n_extremos']
show(top_paises, **ITABLE_KW)

## 3. Ingeniería de Variables

In [ ]:
class FeatureConfig:
    MA_WINDOWS  = [4, 12, 26]
    VOL_WINDOW  = 26
    LAG_ORDERS  = [1, 2, 3, 4]
    MIN_LOG     = 0.01

class PanelOps:
    def __init__(self, frame, group_col='country'):
        self.G = frame.groupby(group_col, observed=True)
    def diff(self, s):  return self.G[s].diff()
    def shift(self, s, lag): return self.G[s].shift(lag)
    def roll(self, s, w, fn='mean'):
        r = self.G[s].rolling(w, min_periods=max(1, w//2))
        return getattr(r, fn)().reset_index(level=0, drop=True)

def add_features(df):
    df = df.sort_values(['country','date']).copy()
    ops = PanelOps(df)

    # Log prices
    for c in ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter','brent_crude_usd']:
        df[f'log_{c}'] = np.log(df[c].clip(lower=FeatureConfig.MIN_LOG))

    # Log-differences (returns)
    df['dlog_petrol'] = ops.diff('log_petrol_usd_liter')
    df['dlog_brent']  = ops.diff('log_brent_crude_usd')

    # Mork asymmetric decomposition
    df['dlog_brent_pos'] = df['dlog_brent'].clip(lower=0)
    df['dlog_brent_neg'] = df['dlog_brent'].clip(upper=0)

    # Lagged brent
    for k in FeatureConfig.LAG_ORDERS:
        df[f'dlog_brent_l{k}'] = ops.shift('dlog_brent', k)

    # Rolling MAs on petrol
    for w in FeatureConfig.MA_WINDOWS:
        df[f'ma{w}'] = ops.roll('petrol_usd_liter', w, 'mean')

    # Rolling volatility
    df['vol_roll26'] = ops.roll('dlog_petrol', FeatureConfig.VOL_WINDOW, 'std') * np.sqrt(52)

    return df

df = add_features(df)
print(f'Nuevas columnas creadas: {[c for c in df.columns if c not in df_main.columns]}')
print(f'Shape final: {df.shape}')

### 3.1 Verificación de la descomposición Mork

In [ ]:
sample = df[['date','country','dlog_brent','dlog_brent_pos','dlog_brent_neg']].dropna().head(8).copy()
sample['suma_check'] = sample['dlog_brent_pos'] + sample['dlog_brent_neg']
sample['ok'] = (sample['suma_check'] - sample['dlog_brent']).abs() < 1e-10
show(sample, **ITABLE_KW)
print('Descomposicion correcta:', sample['ok'].all())

## 4. Análisis Exploratorio de Datos
### 4.1 Estadísticas descriptivas

In [ ]:
num_cols = ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter',
            'brent_crude_usd','tax_percentage','ovx','dxy','gpr']
desc = (
    df[num_cols].agg(['count','mean','median','std','min','max']).T
    .assign(rango=lambda x: x['max']-x['min'],
            cv=lambda x: (x['std']/x['mean']).round(4))
    .round(4).reset_index().rename(columns={'index':'variable'})
)
show(desc, **ITABLE_KW)

### 4.2 Evolución temporal de precios globales

In [ ]:
weekly_global = df.groupby('date', observed=True)[
    ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter','brent_crude_usd']
].mean().reset_index()

fig = make_subplots(specs=[[{'secondary_y': True}]])

for col, name, color in [
    ('petrol_usd_liter',  'Gasolina', COLOR_SEQ[0]),
    ('diesel_usd_liter',  'Diesel',   COLOR_SEQ[1]),
    ('lpg_usd_liter',     'GLP',      COLOR_SEQ[2]),
]:
    fig.add_trace(go.Scatter(x=weekly_global['date'], y=weekly_global[col],
                             name=name, line=dict(color=color, width=2)), secondary_y=False)

fig.add_trace(go.Scatter(x=weekly_global['date'], y=weekly_global['brent_crude_usd'],
                         name='Brent (eje der.)', line=dict(color='gray', dash='dot', width=1.5)),
              secondary_y=True)

add_crisis_bands(fig)
fig.update_layout(template=TEMPLATE, title='Promedio global semanal de precios retail y Brent',
                  height=460, legend=dict(orientation='h', y=-0.15))
fig.update_yaxes(title_text='USD / litro', secondary_y=False)
fig.update_yaxes(title_text='Brent USD / barril', secondary_y=True)
fig.show()

### 4.3 Variables exógenas (OVX, DXY, GPR)

In [ ]:
exog_weekly = df.groupby('date', observed=True)[['ovx','dxy','gpr']].first().reset_index()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['OVX — Volatilidad implícita del crudo',
                                    'DXY — Índice del dólar estadounidense',
                                    'GPR — Riesgo geopolítico'])
pairs = [('ovx', COLOR_SEQ[0]), ('dxy', COLOR_SEQ[1]), ('gpr', COLOR_SEQ[2])]
for row, (col, color) in enumerate(pairs, 1):
    fig.add_trace(go.Scatter(x=exog_weekly['date'], y=exog_weekly[col],
                             line=dict(color=color, width=1.5), showlegend=False), row=row, col=1)
add_crisis_bands(fig)
fig.update_layout(template=TEMPLATE, height=600, title='Series exógenas semanales')
fig.show()

### 4.4 Distribuciones por nivel de ingreso y subsidio

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Por nivel de ingreso', 'Por nivel de subsidio'])

for lvl, color in zip(INCOME_ORDER, COLOR_SEQ):
    sub = df[df['income_level'].astype(str) == lvl]['petrol_usd_liter'].dropna()
    fig.add_trace(go.Box(y=sub, name=lvl, marker_color=color, boxmean=True), row=1, col=1)

for lvl, color in zip(SUBSIDY_ORDER, COLOR_SEQ):
    sub = df[df['subsidy_level'].astype(str) == lvl]['petrol_usd_liter'].dropna()
    fig.add_trace(go.Box(y=sub, name=lvl, marker_color=color, boxmean=True), row=1, col=2)

fig.update_layout(template=TEMPLATE, showlegend=False, height=420,
                  title='Precio de gasolina: distribucion por grupos',
                  yaxis_title='USD/litro', yaxis2_title='USD/litro')
fig.show()

### 4.5 Evolución de precios por región

In [ ]:
regional = df.groupby(['date','region'], observed=True)['petrol_usd_liter'].mean().reset_index()

fig = px.line(regional, x='date', y='petrol_usd_liter', color='region',
              template=TEMPLATE, color_discrete_sequence=COLOR_SEQ,
              title='Precio promedio de gasolina por region',
              labels={'petrol_usd_liter':'USD/litro','date':'Fecha','region':'Region'})
add_crisis_bands(fig)
fig.update_layout(height=460, legend=dict(orientation='h', y=-0.18))
fig.show()

### 4.6 Matriz de correlaciones

In [ ]:
corr_cols = ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter',
             'brent_crude_usd','tax_percentage','ovx','dxy','gpr']
corr_matrix = df[corr_cols].corr().round(3)

fig = px.imshow(corr_matrix, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Matriz de correlaciones — variables numéricas',
                template=TEMPLATE)
fig.update_layout(height=480)
fig.show()

### 4.7 Distribución de retornos semanales de gasolina

In [ ]:
returns = df['dlog_petrol'].dropna()
mu, sigma = returns.mean(), returns.std()
x_range = np.linspace(returns.min(), returns.max(), 300)

fig = go.Figure()
fig.add_trace(go.Histogram(x=returns, nbinsx=80, histnorm='probability density',
                           name='Retornos observados', marker_color=COLOR_SEQ[0], opacity=0.7))
fig.add_trace(go.Scatter(x=x_range, y=norm.pdf(x_range, mu, sigma),
                         mode='lines', name='Normal teorica',
                         line=dict(color='crimson', width=2, dash='dash')))
fig.update_layout(template=TEMPLATE, height=400,
                  title=f'Distribucion de retornos log semanales (gasolina)  mu={mu:.4f}, sigma={sigma:.4f}',
                  xaxis_title='dlog petrol', yaxis_title='Densidad')
fig.show()

sk = stats.skew(returns)
ku = stats.kurtosis(returns)
jb_stat, jb_p = stats.jarque_bera(returns)
print(f'Asimetria: {sk:.4f}  |  Exceso de curtosis: {ku:.4f}')
print(f'Jarque-Bera: stat={jb_stat:.2f}, p={jb_p:.4f}')

## 5. Análisis Estadístico — Preguntas de Investigación

### PI-1: ¿Difieren los precios según nivel de ingreso y subsidio?

In [ ]:
def compare_means(df, group_col, target='petrol_usd_liter'):
    sub = df[[group_col, target]].dropna().copy()
    desc = (sub.groupby(group_col, observed=True)[target]
               .agg(['count','mean','std','min','median','max'])
               .assign(cv=lambda x: x['std']/x['mean'])
               .round(4).reset_index())
    groups = [g[target].values for _, g in sub.groupby(group_col, observed=True)]
    welch  = anova_oneway(groups, use_var='unequal')
    tukey_raw = pairwise_tukeyhsd(sub[target], sub[group_col].astype(str)).summary()
    tukey_data = tukey_raw.data[1:]
    tukey = pd.DataFrame(tukey_data, columns=tukey_raw.data[0])
    return desc, welch, tukey

print('=== PI-1 por INCOME_LEVEL ===')
desc_inc, welch_inc, tukey_inc = compare_means(df, 'income_level')
show(desc_inc, **ITABLE_KW)
print(f'Welch ANOVA: F={welch_inc.statistic:.4f}, p={welch_inc.pvalue:.2e}')

In [ ]:
print('Tukey HSD — income_level:')
show(tukey_inc, **ITABLE_KW)

In [ ]:
print('=== PI-1 por SUBSIDY_LEVEL ===')
desc_sub, welch_sub, tukey_sub = compare_means(df, 'subsidy_level')
show(desc_sub, **ITABLE_KW)
print(f'Welch ANOVA: F={welch_sub.statistic:.4f}, p={welch_sub.pvalue:.2e}')
print()
print('Tukey HSD — subsidy_level:')
show(tukey_sub, **ITABLE_KW)

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Por nivel de ingreso', 'Por nivel de subsidio'])
for lvl, color in zip(INCOME_ORDER, COLOR_SEQ):
    sub = df[df['income_level'].astype(str) == lvl]['petrol_usd_liter'].dropna()
    fig.add_trace(go.Violin(y=sub, name=lvl, fillcolor=color, line_color=color,
                            opacity=0.7, box_visible=True, meanline_visible=True), row=1, col=1)
for lvl, color in zip(SUBSIDY_ORDER, COLOR_SEQ):
    sub = df[df['subsidy_level'].astype(str) == lvl]['petrol_usd_liter'].dropna()
    fig.add_trace(go.Violin(y=sub, name=lvl, fillcolor=color, line_color=color,
                            opacity=0.7, box_visible=True, meanline_visible=True), row=1, col=2)
fig.update_layout(template=TEMPLATE, height=460, showlegend=False,
                  title='PI-1: Distribucion de precios por grupo',
                  yaxis_title='USD/litro', yaxis2_title='USD/litro')
fig.show()

### PI-2: ¿Reducen los subsidios la volatilidad?

In [ ]:
vol_por_pais = (
    df.groupby(['country','subsidy_level'], observed=True)['dlog_petrol']
      .std().mul(np.sqrt(52)).reset_index(name='vol_anual')
)
vol_summary = (
    vol_por_pais.groupby('subsidy_level', observed=True)['vol_anual']
                .agg(['count','mean','median','std','min','max'])
                .round(4).reset_index()
)
show(vol_summary, **ITABLE_KW)

# Levene test
vol_groups = [vol_por_pais.loc[vol_por_pais['subsidy_level'].astype(str) == lvl, 'vol_anual'].dropna().values
              for lvl in SUBSIDY_ORDER]
vol_groups = [g for g in vol_groups if len(g) > 1]
lev_stat, lev_p = levene(*vol_groups, center='median')
print(f'Test de Levene (Brown-Forsythe): stat={lev_stat:.4f}, p={lev_p:.4f}')

In [ ]:
fig = px.violin(vol_por_pais, x='subsidy_level', y='vol_anual',
                color='subsidy_level', category_orders={'subsidy_level': SUBSIDY_ORDER},
                box=True, points='all',
                color_discrete_sequence=COLOR_SEQ,
                template=TEMPLATE,
                title='PI-2: Volatilidad anualizada por nivel de subsidio',
                labels={'vol_anual':'Volatilidad anual (desvio log)','subsidy_level':'Nivel de subsidio'})
fig.update_layout(height=440, showlegend=False)
fig.show()

### PI-3: Pass-through del Brent al precio retail

In [ ]:
def passthrough_symmetric(df, n_lags=4):
    lag_cols = [f'dlog_brent_l{k}' for k in range(1, n_lags+1)]
    needed   = ['dlog_petrol','dlog_brent'] + lag_cols
    panel_df = df[['country','date'] + needed].dropna().copy()
    panel    = panel_df.set_index(['country','date']).sort_index()
    exog_cols = ['dlog_brent'] + lag_cols
    model    = PanelOLS(panel['dlog_petrol'], panel[exog_cols],
                        entity_effects=True, time_effects=False)
    res      = model.fit(cov_type='kernel', kernel='bartlett', bandwidth=4)
    coef_df  = pd.DataFrame({'coef': res.params, 'se': res.std_errors, 'pval': res.pvalues}).reset_index()
    coef_df.columns = ['variable','coef','se','pval']
    coef_df['cumsum'] = coef_df['coef'].cumsum()
    return coef_df, res

coef_sym, res_sym = passthrough_symmetric(df)
print('Pass-through simetrico (efectos fijos por pais):')
show(coef_sym.round(6), **ITABLE_KW)
print(f'R² within = {res_sym.rsquared:.4f}')

In [ ]:
def passthrough_country(df, n_lags=2):
    lag_cols = [f'dlog_brent_l{k}' for k in range(1, n_lags+1)]
    needed   = ['dlog_petrol','dlog_brent'] + lag_cols
    results  = []
    for country, grp in df.groupby('country', observed=True):
        sub = grp[needed + ['subsidy_level']].dropna()
        if len(sub) < 20:
            continue
        X = sm.add_constant(sub[['dlog_brent'] + lag_cols])
        try:
            ols = sm.OLS(sub['dlog_petrol'], X).fit(cov_type='HAC', cov_kwds={'maxlags':4})
            results.append({
                'country':     str(country),
                'beta0':       ols.params.get('dlog_brent', np.nan),
                'pval_beta0':  ols.pvalues.get('dlog_brent', np.nan),
                'r2':          ols.rsquared,
                'n':           len(sub),
                'subsidy_level': str(sub['subsidy_level'].mode()[0]),
            })
        except Exception:
            pass
    return pd.DataFrame(results).sort_values('beta0', ascending=False)

cbc = passthrough_country(df)
print(f'Pass-through pais a pais: {len(cbc)} paises estimados')
show(cbc.round(4).head(20), **ITABLE_KW)

In [ ]:
d = cbc.sort_values('beta0', ascending=True)
colors = ['crimson' if p >= 0.05 else COLOR_SEQ[0] for p in d['pval_beta0']]

fig = go.Figure(go.Bar(
    x=d['beta0'], y=d['country'], orientation='h',
    marker_color=colors,
    text=d['beta0'].round(3), textposition='outside',
))
fig.update_layout(
    template=TEMPLATE, height=900,
    title='PI-3: Pass-through del Brent por pais (beta contemporaneo)',
    xaxis_title='beta_0 (dlog_brent)', yaxis_title='',
    yaxis=dict(tickfont=dict(size=8)),
    shapes=[dict(type='line', x0=0, x1=0, y0=-0.5, y1=len(d)-0.5,
                 line=dict(color='black', dash='dot', width=1))],
    annotations=[dict(x=1.05, y=1, xref='paper', yref='paper',
                      text='<span style="color:crimson">■</span> no signif (p>=0.05)',
                      showarrow=False, font=dict(size=10))]
)
fig.show()

## 6. Análisis de Crisis: COVID-19 vs Guerra de Ucrania

In [ ]:
period_order = ['Pre-COVID','COVID-19','Inter-periodo','Guerra Ucrania','Post-guerra']

def assign_period(date):
    if date < COVID_START:            return 'Pre-COVID'
    if COVID_START <= date <= COVID_END:   return 'COVID-19'
    if COVID_END < date < UKRAINE_START:   return 'Inter-periodo'
    if UKRAINE_START <= date <= UKRAINE_END: return 'Guerra Ucrania'
    return 'Post-guerra'

df['crisis_period'] = df['date'].map(assign_period)
print(df['crisis_period'].value_counts())

In [ ]:
weekly_crisis = df.groupby('date', observed=True)[
    ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter','brent_crude_usd']
].mean().reset_index()

fig = make_subplots(rows=2, cols=2, shared_xaxes=True,
                    subplot_titles=['Gasolina','Diesel','GLP','Brent (USD/barril)'])
cols_crisis = ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter','brent_crude_usd']
positions   = [(1,1),(1,2),(2,1),(2,2)]
for (r,c), col, color in zip(positions, cols_crisis, COLOR_SEQ):
    fig.add_trace(go.Scatter(x=weekly_crisis['date'], y=weekly_crisis[col],
                             line=dict(color=color, width=2), showlegend=False), row=r, col=c)
add_crisis_bands(fig)
fig.update_layout(template=TEMPLATE, height=500,
                  title='Evolucion de precios durante periodos de crisis')
fig.show()

In [ ]:
price_cols = ['petrol_usd_liter','diesel_usd_liter','lpg_usd_liter','brent_crude_usd']
country_means = (df.groupby(['country','crisis_period'], observed=True)[price_cols]
                   .mean().reset_index())

period_stats = []
for period in period_order:
    sub = country_means[country_means['crisis_period'] == period]
    for col in price_cols:
        period_stats.append({
            'periodo': period,
            'combustible': col.replace('_usd_liter','').replace('_usd',''),
            'media': sub[col].mean(),
            'mediana': sub[col].median(),
            'std': sub[col].std(),
            'n_paises': len(sub),
        })
stats_df = pd.DataFrame(period_stats).round(4)
show(stats_df, **ITABLE_KW)

In [ ]:
# Mann-Whitney: COVID vs Guerra Ucrania (gasolina)
covid_g     = country_means.loc[country_means['crisis_period']=='COVID-19',        'petrol_usd_liter'].dropna()
ukraine_g   = country_means.loc[country_means['crisis_period']=='Guerra Ucrania', 'petrol_usd_liter'].dropna()
pre_covid_g = country_means.loc[country_means['crisis_period']=='Pre-COVID',      'petrol_usd_liter'].dropna()

for a_name, a_vals, b_name, b_vals in [
    ('COVID-19', covid_g, 'Pre-COVID', pre_covid_g),
    ('Guerra Ucrania', ukraine_g, 'Pre-COVID', pre_covid_g),
    ('COVID-19', covid_g, 'Guerra Ucrania', ukraine_g),
]:
    stat, p = mannwhitneyu(a_vals, b_vals, alternative='two-sided')
    print(f'{a_name} vs {b_name}: U={stat:.0f}, p={p:.4f}')

In [ ]:
vol_crisis = (
    df.groupby(['country','crisis_period'], observed=True)['dlog_petrol']
      .std().mul(np.sqrt(52)).reset_index(name='vol_anual')
)
fig = px.box(vol_crisis, x='crisis_period', y='vol_anual',
             category_orders={'crisis_period': period_order},
             color='crisis_period', color_discrete_sequence=COLOR_SEQ,
             template=TEMPLATE, points='all',
             title='Volatilidad anualizada por periodo de crisis',
             labels={'vol_anual':'Volatilidad anual','crisis_period':'Periodo'})
fig.update_layout(height=430, showlegend=False)
fig.show()

## 7. Modelado Predictivo
### 7.1 Preparación de datos

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODEL_FEATS = ['brent_crude_usd','tax_percentage','ovx','dxy','gpr','ma4','ma12','dlog_brent',
               'income_level','subsidy_level']
TARGET = 'petrol_usd_liter'

model_df = df[['date', TARGET] + MODEL_FEATS].dropna().copy()
unique_dates = np.array(sorted(model_df['date'].unique()))
split_date   = pd.Timestamp(unique_dates[int(len(unique_dates) * 0.8) - 1])
train_df = model_df[model_df['date'] <= split_date].copy()
test_df  = model_df[model_df['date'] >  split_date].copy()

print(f'Train: {len(train_df):,} obs  ({train_df.date.min().date()} — {train_df.date.max().date()})')
print(f'Test : {len(test_df):,}  obs  ({test_df.date.min().date()} — {test_df.date.max().date()})')

num_feats = ['brent_crude_usd','tax_percentage','ovx','dxy','gpr','ma4','ma12','dlog_brent']
cat_feats = ['income_level','subsidy_level']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_feats),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_feats),
])

X_train = train_df[MODEL_FEATS]
y_train = train_df[TARGET]
X_test  = test_df[MODEL_FEATS]
y_test  = test_df[TARGET]

### 7.2 Entrenamiento de modelos (Ridge, Árbol de Decisión, HistGradientBoosting)

In [ ]:
models = {
    'Ridge':                Pipeline([('pre', preprocessor), ('mdl', Ridge(alpha=1.0))]),
    'Decision Tree':        Pipeline([('pre', preprocessor), ('mdl', DecisionTreeRegressor(max_depth=6, random_state=42))]),
    'HistGradientBoosting': Pipeline([('pre', preprocessor), ('mdl', HistGradientBoostingRegressor(max_iter=200, random_state=42))]),
}

results = {}
preds   = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_hat = pipe.predict(X_test)
    preds[name] = y_hat
    results[name] = {
        'RMSE': mean_squared_error(y_test, y_hat, squared=False),
        'MAE':  mean_absolute_error(y_test, y_hat),
        'R2':   r2_score(y_test, y_hat),
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.4f}  MAE={results[name]["MAE"]:.4f}  R2={results[name]["R2"]:.4f}')

metrics_df = pd.DataFrame(results).T.reset_index().rename(columns={'index':'Modelo'}).round(4)
show(metrics_df, **ITABLE_KW)

### 7.3 Actual vs Predicho

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=list(models.keys()))
for i, (name, y_hat) in enumerate(preds.items(), 1):
    fig.add_trace(go.Scatter(x=y_test, y=y_hat, mode='markers',
                             marker=dict(color=COLOR_SEQ[i-1], size=3, opacity=0.5),
                             name=name, showlegend=False), row=1, col=i)
    lims = [min(y_test.min(), y_hat.min()), max(y_test.max(), y_hat.max())]
    fig.add_trace(go.Scatter(x=lims, y=lims, mode='lines',
                             line=dict(color='black', dash='dot', width=1),
                             showlegend=False), row=1, col=i)
fig.update_layout(template=TEMPLATE, height=380,
                  title='Comparacion: valores reales vs predichos en el conjunto de prueba')
fig.show()

### 7.4 Importancia de variables (HistGradientBoosting)

In [ ]:
hgb_pipe = models['HistGradientBoosting']
hgb_mdl  = hgb_pipe.named_steps['mdl']
pre      = hgb_pipe.named_steps['pre']
cat_names = list(pre.named_transformers_['cat']
                    .get_feature_names_out(cat_feats))
feat_names = num_feats + cat_names

importances = pd.Series(hgb_mdl.feature_importances_, index=feat_names)                    .sort_values(ascending=True)

fig = go.Figure(go.Bar(x=importances.values, y=importances.index,
                       orientation='h', marker_color=COLOR_SEQ[2]))
fig.update_layout(template=TEMPLATE, height=400,
                  title='Importancia de variables — HistGradientBoosting',
                  xaxis_title='Importancia relativa', yaxis_title='')
fig.show()

## 8. Chile en el Contexto Sudamericano

In [ ]:
SOUTH_AMERICA = ['Argentina','Brazil','Chile','Colombia','Ecuador','Peru','Venezuela']
sa_df = df[df['country'].astype(str).isin(SOUTH_AMERICA)].copy()
sa_en_dataset = sa_df['country'].astype(str).unique().tolist()
print(f'Paises sudamericanos en el dataset: {sorted(sa_en_dataset)}')

In [ ]:
sa_weekly = sa_df.groupby(['date','country'], observed=True)['petrol_usd_liter'].mean().reset_index()
fig = px.line(sa_weekly, x='date', y='petrol_usd_liter', color='country',
              template=TEMPLATE, color_discrete_sequence=COLOR_SEQ,
              title='Precio de gasolina en Sudamerica: foco en Chile',
              labels={'petrol_usd_liter':'USD/litro','date':'Fecha','country':'Pais'})
add_crisis_bands(fig)
fig.update_layout(height=460, legend=dict(orientation='h', y=-0.2))
fig.show()

In [ ]:
sa_stats = []
for country in sorted(sa_en_dataset):
    sub = sa_df[sa_df['country'].astype(str) == country]
    sa_stats.append({
        'pais':          country,
        'n_obs':         len(sub),
        'precio_medio':  sub['petrol_usd_liter'].mean(),
        'precio_mediana':sub['petrol_usd_liter'].median(),
        'precio_std':    sub['petrol_usd_liter'].std(),
        'precio_min':    sub['petrol_usd_liter'].min(),
        'precio_max':    sub['petrol_usd_liter'].max(),
        'vol_anual':     sub['dlog_petrol'].std() * np.sqrt(52),
    })
sa_summary = pd.DataFrame(sa_stats).round(4)
show(sa_summary, **ITABLE_KW)

In [ ]:
sa_sorted = sa_summary.sort_values('vol_anual', ascending=True)
colors_sa = ['crimson' if p == 'Chile' else COLOR_SEQ[0] for p in sa_sorted['pais']]

fig = go.Figure(go.Bar(x=sa_sorted['pais'], y=sa_sorted['vol_anual'],
                       marker_color=colors_sa,
                       text=sa_sorted['vol_anual'].round(3), textposition='outside'))
fig.update_layout(template=TEMPLATE, height=400,
                  title='Volatilidad anualizada — paises sudamericanos',
                  xaxis_title='Pais', yaxis_title='Volatilidad anual')
fig.show()

In [ ]:
pt_sa = passthrough_country(sa_df, n_lags=2).sort_values('beta0', ascending=False)
show(pt_sa.round(4), **ITABLE_KW)

colors_pt = ['crimson' if p == 'Chile' else
             (COLOR_SEQ[1] if pv < 0.05 else '#cccccc')
             for p, pv in zip(pt_sa['country'], pt_sa['pval_beta0'])]
fig = go.Figure(go.Bar(x=pt_sa['country'], y=pt_sa['beta0'],
                       marker_color=colors_pt,
                       text=pt_sa['beta0'].round(3), textposition='outside'))
fig.update_layout(template=TEMPLATE, height=400,
                  title='Pass-through del Brent — paises sudamericanos (beta_0)',
                  xaxis_title='Pais', yaxis_title='beta_0')
fig.show()

## 9. Conclusiones

### Síntesis de hallazgos

**PI-1 — Diferencias por ingreso y subsidio.**  
Los precios de gasolina difieren significativamente entre niveles de ingreso y esquemas de subsidio (Welch ANOVA, p < 0.001). Los países de alto ingreso muestran precios más altos en términos absolutos, mientras que los países con subsidio alto presentan precios artificialmente bajos. Los pares de Tukey confirman diferencias estadísticamente significativas en todas las combinaciones relevantes.

**PI-2 — Subsidios y volatilidad.**  
El test de Levene muestra que la varianza de la volatilidad anualizada no es homogénea entre grupos de subsidio. Sin embargo, mayor subsidio no implica necesariamente menor volatilidad. Los países con subsidio muy alto tienen dispersión interna elevada, sugiriendo que la política de subsidio es heterogénea en su implementación.

**PI-3 — Pass-through del Brent.**  
El modelo de panel OLS con efectos fijos por país confirma un pass-through positivo y significativo del Brent al precio retail. El coeficiente contemporáneo beta_0 varía fuertemente entre países (0 a >1), reflejando diferencias en política energética, impuestos y grado de control de precios. Los países con subsidio alto tienden a presentar menor pass-through.

**Modelado predictivo.**  
HistGradientBoosting supera a Ridge y al árbol de decisión en todas las métricas. El Brent, OVX y las medias móviles son las variables más importantes. La mayoría de modelos alcanzan MAE < 0.05 USD/litro en el conjunto de prueba.

**Chile en Sudamérica.**  
Chile exhibe un pass-through relativamente alto y una volatilidad moderada en comparación con sus pares sudamericanos. Argentina y Venezuela presentan patrones atípicos por sus esquemas de control de cambio y subsidio.

### Limitaciones

- Las series FRED externas tienen cobertura parcial en el período 2020–2026.
- La frecuencia semanal del panel puede enmascarar dinámicas intra-semanales.
- Los modelos predictivos no capturan choques de política idiosincráticos por país.

### Referencias

- Global Fuel Prices 2020–2026 (Kaggle, Belbino)  
- FRED — Federal Reserve Bank of St. Louis  
- Caldara & Iacoviello (2022) — Geopolitical Risk Index  
- Mork, K. A. (1989) — Oil and the Macroeconomy When Prices Go Up and Down  
